In [1]:
import pandas as pd
import numpy as np

# Apenas para simular a extração do sistema legado da empresa
np.random.seed(42)
n = 1500

dados = {
    'ID_Pedido': [f'PED{i:04d}' for i in range(1, n+1)],
    'DataPedido': pd.date_range(start='2026-01-01', periods=n, freq='h'),
    'DataEntrega': pd.date_range(start='2026-01-03', periods=n, freq='h') + pd.to_timedelta(np.random.randint(-2, 12, size=n), unit='d'),
    'Categoria': np.random.choice(['Eletrônicos', 'Móveis', 'Vestuário', 'Esportes'], size=n),
    'PrecoUnitario': np.random.choice([150.0, 350.0, 1200.0, 80.0, np.nan], size=n, p=[0.3, 0.3, 0.2, 0.1, 0.1]),
    'Quantidade': np.random.choice([1, 2, 3, 4], size=n),
    'NotaCliente': np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.05, 0.05, 0.2, 0.3, 0.2, 0.2]),
    'TipoFrete': np.random.choice(['Expresso', 'Normal', 'Econômico'], size=n),
    'RegiaoEntrega': np.random.choice(['Sul', 'Sudeste', 'Nordeste', 'Centro-Oeste', np.nan], size=n, p=[0.3, 0.4, 0.1, 0.1, 0.1])
}

df_bruto = pd.DataFrame(dados)
df_bruto.to_csv('omnistore_vendas.csv', index=False)
print("Ficheiro 'omnistore_vendas.csv' gerado com sucesso!")

display(df_bruto)

Ficheiro 'omnistore_vendas.csv' gerado com sucesso!


C:\Users\Pichau\AppData\Local\Temp\ipykernel_13516\1097024065.py:11: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  'DataEntrega': pd.date_range(start='2026-01-03', periods=n, freq='h') + pd.to_timedelta(np.random.randint(-2, 12, size=n), unit='d'),


,ID_Pedido,DataPedido,DataEntrega,Categoria,PrecoUnitario,Quantidade,NotaCliente,TipoFrete,RegiaoEntrega
0,PED0001,2026-01-01 00:00:00,2026-01-07 00:00:00,Eletrônicos,350.0,1,2.0,Expresso,Sudeste
1,PED0002,2026-01-01 01:00:00,2026-01-04 01:00:00,Eletrônicos,150.0,3,4.0,Normal,Sudeste
2,PED0003,2026-01-01 02:00:00,2026-01-13 02:00:00,Eletrônicos,80.0,1,4.0,Expresso,Centro-Oeste
3,PED0004,2026-01-01 03:00:00,2026-01-11 03:00:00,Eletrônicos,150.0,3,4.0,Expresso,Centro-Oeste
4,PED0005,2026-01-01 04:00:00,2026-01-08 04:00:00,Móveis,350.0,2,NaN,Normal,Centro-Oeste
...,...,...,...,...,...,...,...,...,...
1495,PED1496,2026-03-04 07:00:00,2026-03-14 07:00:00,Móveis,150.0,3,5.0,Normal,nan
1496,PED1497,2026-03-04 08:00:00,2026-03-11 08:00:00,Eletrônicos,1200.0,3,NaN,Expresso,Centro-Oeste
1497,PED1498,2026-03-04 09:00:00,2026-03-06 09:00:00,Vestuário,NaN,3,3.0,Normal,Sudeste
1498,PED1499,2026-03-04 10:00:00,2026-03-17 10:00:00,Esportes,350.0,4,4.0,Econômico,Sudeste


In [9]:
df_bruto['PrecoUnitario'] = df_bruto['PrecoUnitario'].fillna(
    df_bruto.groupby('Categoria')['PrecoUnitario'].transform('median'))

pd.to_datetime(df_bruto['DataPedido'])
pd.to_datetime(df_bruto['DataEntrega'])

df_bruto['Quantidade'] = df_bruto['Quantidade'].astype(int)

df_bruto['FaturamentoTotal'] = df_bruto['Quantidade'] * df_bruto['PrecoUnitario']

df_bruto['DiasDeEntrega'] = df_bruto['DataEntrega'] - df_bruto['DataPedido']

df_bruto['ATRASADOS'] = df_bruto['DiasDeEntrega'].dt.days > 7
df_bruto['ATRASADOS'] = df_bruto['ATRASADOS'].astype(int)

df_bruto['TaxaGlobal'] = df_bruto['ATRASADOS'].mean() * 100


print(df_bruto.groupby('TipoFrete')['ATRASADOS'].mean() * 100)


print('O ranking é:\n', df_bruto.groupby('Categoria')['FaturamentoTotal'].sum().nlargest(3))

print(df_bruto.groupby('RegiaoEntrega')['FaturamentoTotal'].sum())

display(df_bruto)
print(df_bruto.info())

TipoFrete
Econômico    43.407708
Expresso     39.708940
Normal       43.155894
Name: ATRASADOS, dtype: float64
O ranking é:
 Categoria
Eletrônicos    450190.0
Vestuário      411070.0
Esportes       409740.0
Name: FaturamentoTotal, dtype: float64
RegiaoEntrega
Centro-Oeste    166940.0
Nordeste        159430.0
Sudeste         705090.0
Sul             489850.0
nan             151700.0
Name: FaturamentoTotal, dtype: float64


,ID_Pedido,DataPedido,DataEntrega,Categoria,PrecoUnitario,Quantidade,NotaCliente,TipoFrete,RegiaoEntrega,FaturamentoTotal,DiasDeEntrega,ATRASADOS,TaxaGlobal
0,PED0001,2026-01-01 00:00:00,2026-01-07 00:00:00,Eletrônicos,350.0,1,2.0,Expresso,Sudeste,350.0,6 days,0,42.133333
1,PED0002,2026-01-01 01:00:00,2026-01-04 01:00:00,Eletrônicos,150.0,3,4.0,Normal,Sudeste,450.0,3 days,0,42.133333
2,PED0003,2026-01-01 02:00:00,2026-01-13 02:00:00,Eletrônicos,80.0,1,4.0,Expresso,Centro-Oeste,80.0,12 days,1,42.133333
3,PED0004,2026-01-01 03:00:00,2026-01-11 03:00:00,Eletrônicos,150.0,3,4.0,Expresso,Centro-Oeste,450.0,10 days,1,42.133333
4,PED0005,2026-01-01 04:00:00,2026-01-08 04:00:00,Móveis,350.0,2,NaN,Normal,Centro-Oeste,700.0,7 days,0,42.133333
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,PED1496,2026-03-04 07:00:00,2026-03-14 07:00:00,Móveis,150.0,3,5.0,Normal,nan,450.0,10 days,1,42.133333
1496,PED1497,2026-03-04 08:00:00,2026-03-11 08:00:00,Eletrônicos,1200.0,3,NaN,Expresso,Centro-Oeste,3600.0,7 days,0,42.133333
1497,PED1498,2026-03-04 09:00:00,2026-03-06 09:00:00,Vestuário,350.0,3,3.0,Normal,Sudeste,1050.0,2 days,0,42.133333
1498,PED1499,2026-03-04 10:00:00,2026-03-17 10:00:00,Esportes,350.0,4,4.0,Econômico,Sudeste,1400.0,13 days,1,42.133333


<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype          
---  ------            --------------  -----          
 0   ID_Pedido         1500 non-null   str            
 1   DataPedido        1500 non-null   datetime64[us] 
 2   DataEntrega       1500 non-null   datetime64[us] 
 3   Categoria         1500 non-null   str            
 4   PrecoUnitario     1500 non-null   float64        
 5   Quantidade        1500 non-null   int64          
 6   NotaCliente       1205 non-null   float64        
 7   TipoFrete         1500 non-null   str            
 8   RegiaoEntrega     1500 non-null   str            
 9   FaturamentoTotal  1500 non-null   float64        
 10  DiasDeEntrega     1500 non-null   timedelta64[us]
 11  ATRASADOS         1500 non-null   int64          
 12  TaxaGlobal        1500 non-null   float64        
dtypes: datetime64[us](2), float64(4), int64(2), str(4), timedelta64[us](1)
mem

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_rows = 500

tipos_produto = ['Eletrônicos', 'Vestuário', 'Alimentos', 'Livros']
regioes = ['Sudeste', 'Sul', 'Nordeste', 'Centro-Oeste']
cidades_por_regiao = {
    'Sudeste': ['São Paulo', 'Rio de Janeiro', 'Belo Horizonte'],
    'Sul': ['Porto Alegre', 'Curitiba', 'Florianópolis'],
    'Nordeste': ['Salvador', 'Recife', 'Fortaleza'],
    'Centro-Oeste': ['Brasília', 'Goiânia', 'Cuiabá']
}

tipo_produto = np.random.choice(tipos_produto, size=n_rows)
regiao = np.random.choice(regioes, size=n_rows)
cidade_destino = [np.random.choice(cidades_por_regiao[r]) for r in regiao]

# Introduzir nulos em CidadeDestino
cidade_destino = [np.nan if np.random.rand() < 0.08 else c for c in cidade_destino]

peso_pacote = np.random.uniform(0.5, 25.0, size=n_rows)
# Introduzir nulos em PesoPacote
peso_pacote = [np.nan if np.random.rand() < 0.10 else p for p in peso_pacote]

data_postagem = pd.date_range(start='2026-01-01', periods=n_rows, freq='h')
dias_transito = np.random.randint(1, 12, size=n_rows)
data_entrega = data_postagem + pd.to_timedelta(dias_transito, unit='days')

data_postagem_str = data_postagem.strftime('%Y-%m-%d')
data_entrega_str = data_entrega.strftime('%Y-%m-%d')

num_tentativas = np.random.choice([1, 2, 3, 4], size=n_rows, p=[0.7, 0.2, 0.08, 0.02])
num_tentativas_str = [str(x) if np.random.rand() < 0.1 else x for x in num_tentativas]

df = pd.DataFrame({
    'ID_Pedido': [f'PED-{1000+i}' for i in range(n_rows)],
    'TipoProduto': tipo_produto,
    'Regiao': regiao,
    'CidadeDestino': cidade_destino,
    'PesoPacote': peso_pacote,
    'DataPostagem': data_postagem_str,
    'DataEntrega': data_entrega_str,
    'NumeroTentativas': num_tentativas_str
})

df.to_csv('techlog_bruto.csv', index=False)
print("Dataset gerado com sucesso! Linhas:", len(df))
print(df.head())
print("\nValores nulos:")
print(df.isnull().sum())

Dataset gerado com sucesso! Linhas: 500
  ID_Pedido  TipoProduto        Regiao CidadeDestino  PesoPacote DataPostagem  \
0  PED-1000    Alimentos      Nordeste        Recife   18.423943   2026-01-01   
1  PED-1001       Livros       Sudeste           NaN    2.499935   2026-01-01   
2  PED-1002  Eletrônicos       Sudeste     São Paulo    1.978626   2026-01-01   
3  PED-1003    Alimentos  Centro-Oeste      Brasília    6.554029   2026-01-01   
4  PED-1004    Alimentos      Nordeste           NaN    4.408845   2026-01-01   

  DataEntrega NumeroTentativas  
0  2026-01-02                1  
1  2026-01-09                1  
2  2026-01-04                3  
3  2026-01-09                2  
4  2026-01-09                2  

Valores nulos:
ID_Pedido            0
TipoProduto          0
Regiao               0
CidadeDestino       47
PesoPacote          48
DataPostagem         0
DataEntrega          0
NumeroTentativas     0
dtype: int64


In [2]:
df_bruto = pd.read_csv('techlog_bruto.csv')

# 1. Limpeza de dados e tratamento com groupby

df_bruto('PesoPacote') = df_bruto('PesoPacote').fillna(
    df_bruto.groupby('TipoProduto')['PesoPacote'].transform('mean')
)
display(df_bruto)
display(df_bruto.info())

SyntaxError: cannot assign to function call here. Maybe you meant '==' instead of '='? (3899120395.py, line 5)